In [3]:
import numpy as np
import pandas as pd
from scipy import sparse
import os

# =====================================================================
# FUNZIONE 1: COSTRUZIONE DELLA MATRICE DI ADIACENZA (L)
# =====================================================================
def prepara_matrice_adiacenza(sorgenti, destinazioni, n_nodi):
    """
    Costruisce la matrice di adiacenza sparsa L.
    A differenza del PageRank, qui NON dividiamo per i link uscenti.
    Mettiamo semplicemente un '1' se esiste il link da i a j.
    """
    # Creiamo un array di "1" per indicare la presenza dei link
    #len(sorgenti) ci dice quanti link ci sono, e quindi quanti "1" dobbiamo inserire nella matrice
    dati_link = np.ones(len(sorgenti))
    
    # Costruiamo la matrice sparsa L in formato CSR (ottimizzato per le moltiplicazioni riga-colonna)
    # L[i, j] = 1 significa che c'è un link dal nodo i al nodo j
    #shape=(n_nodi, n_nodi) specifica che la matrice è quadrata e ha dimensione n_nodi x n_nodi
    L = sparse.csr_matrix((dati_link, (sorgenti, destinazioni)), shape=(n_nodi, n_nodi))
    
    return L

# ===============================
# FUNZIONE 2: L'ALGORITMO HITS 
# ===============================
def hits_algorithm(L, max_iter=100, tol=1e-8):
    """
    Calcola Hub (y) e Authority (x) usando la matrice sparsa L.
    """
    # Estraiamo il numero totale di nodi dalla matrice L
    n = L.shape[0] 
    
    # 1. Inizializzazione: x (authority) e y (hub) partono da un vettore di tutti 1
    #np.ones(n) crea un array di dimensione n pieno di 1.0
    x = np.ones(n)
    y = np.ones(n)
    
    # Eseguiamo il ciclo iterativo 
    #max_iter è il numero massimo di iterazioni che vogliamo eseguire per cercare la convergenza, fissato a 100
    for i in range(1, max_iter + 1):
        
        # Salviamo i vecchi valori per controllare poi la convergenza 
        #copio i valori di x in x_old per poter confrontare dopo l'aggiornamento e verificare se abbiamo raggiunto la convergenza
        x_old = x.copy()
        
        # 2. Aggiornamento Authority: x = L^T * y , cioè moltiplichiamo la matrice trasposta per il vettore dei hub (come da teoria)
        # L.T è la matrice trasposta. Moltiplichiamo i link entranti per gli Hub
        x = L.T.dot(y)
        
        # 3. Aggiornamento Hub: y = L * x, cioè moltiplichiamo la matrice per il vettore delle authority (come da teoria)
        # Moltiplichiamo i link uscenti per le nuove Authority appena calcolate
        y = L.dot(x)
        
        # 4. Normalizzazione (L1-norm)
        # Dividiamo ogni vettore per la somma dei suoi elementi assoluti, per stabilizzare il sistema
        # Questo è importante per evitare un overflow numerico e per garantire che i valori rimangano in una scala gestibile
        #qui utilizzo la norma 1
        x = x / np.linalg.norm(x, 1)
        y = y / np.linalg.norm(y, 1)
        
        # 5. Criterio di arresto: se i valori non cambiano più, ci fermiamo prima di max_iter
        #Grazie a quesot criterio possiamo fermarci quando i valori di x non cambiano più (con una tolleranza definita da tol)
        #np.linalg.norm(x - x_old, 1) calcola la differenza tra il nuovo vettore x e quello vecchio, 
        # e se questa differenza è minore di tol, consideriamo che abbiamo raggiunto la convergenza
        if np.linalg.norm(x - x_old, 1) < tol:
            print(f"HITS Convergenza raggiunta all'iterazione {i}.")
            return x, y
            
    print("HITS: Attenzione, convergenza non raggiunta.")
    return x, y



In [5]:
import os
import pandas as pd
import numpy as np

# =====================================================================
# TEST HITS SCENARIO 1 CON DATASET 100x100
# =====================================================================

numero_totale_nodi = 100 
print("--- ANALISI HITS: SCENARIO 1 ---")

# 1. Definiamo il file di input 
nome_file = '../DataSet_CasoStudio1/Rete_100/dataset_scenario1.csv'
print(f"Caricamento del dataset: {nome_file}...")

# Verifichiamo che il file esista prima di procedere
if os.path.exists(nome_file):

    # 2. Leggiamo il CSV con Pandas
    #inserisco il dataset in un DataFrame di Pandas per poterlo manipolare più facilmente
    df = pd.read_csv(nome_file)

    # Estraiamo le colonne 'Source' e 'Target' come array numpy per costruire la matrice di adiacenza
    #  df['Source'].values ci dà un array numpy con i valori della colonna 'Source', che rappresentano i nodi di partenza dei link
    #  df['Target'].values ci dà un array numpy con i valori della colonna 'Target', che rappresentano i nodi di arrivo dei link
    nodi_sorgente = df['Source'].values
    nodi_destinazione = df['Target'].values

    # 3. Generiamo la matrice di adiacenza sparsa L (specifica per HITS)
    print("Generazione automatica della matrice di adiacenza L sparsa...")
    L_sparsa = prepara_matrice_adiacenza(nodi_sorgente, nodi_destinazione, numero_totale_nodi)

    # 4. Esecuzione dell'algoritmo HITS
    print("\nAvvio del calcolo HITS...")
    authority_vector, hub_vector = hits_algorithm(L_sparsa, max_iter=1000)

    # Output a schermo (stampiamo solo la Top 5 per brevità visiva)
    print("\nTop 5 AUTHORITY (Chi riceve i link migliori):")

    # Creiamo una lista di tuple (nodo, authority_score) e la ordiniamo in base al punteggio di authority
    classifica_auth = [(i, authority_vector[i]) for i in range(numero_totale_nodi)]
    classifica_auth.sort(key=lambda x: x[1], reverse=True) # Ordiniamo in modo decrescente
    
    # Stampiamo i primi 5 nodi con i punteggi di authority più alti
    for pos, (nodo, auth) in enumerate(classifica_auth[:5]):
        print(f"{pos+1}° Posto -> Nodo {nodo}: Score {auth:.4f}")

    print("\nTop 5 HUB (Chi smista i link migliori):")

    # Creiamo una lista di tuple (nodo, hub_score) e la ordiniamo in base al punteggio di hub
    classifica_hub = [(i, hub_vector[i]) for i in range(numero_totale_nodi)]
    classifica_hub.sort(key=lambda x: x[1], reverse=True) # Ordiniamo in modo decrescente

    # Stampiamo i primi 5 nodi con i punteggi di hub più alti
    for pos, (nodo, hub) in enumerate(classifica_hub[:5]):
        print(f"{pos+1}° Posto -> Nodo {nodo}: Score {hub:.4f}")

    # =====================================================================
    # SALVATAGGIO DEI RISULTATI NELLA CARTELLA 'Risultati_CasoStudio1_HITS'
    # =====================================================================
    percorso_risultati = '../Risultati_CasoStudio1_HITS/Rete_100/'

    # Se la cartella non esiste, creiamola
    if not os.path.exists(percorso_risultati):
        os.makedirs(percorso_risultati)
        print(f"\nCartella creata con successo in: {percorso_risultati}")

    # Creiamo un DataFrame "fuso" che contenga sia Hub che Authority per ogni nodo
    # Partiamo da una lista ordinata numericamente (dal Nodo 0 al Nodo 99)
    risultati_combinati = []

    # Popoliamo la lista con i risultati di Authority e Hub per ogni nodo
    for i in range(numero_totale_nodi):
        risultati_combinati.append({
            'Nodo': i,
            'Authority': authority_vector[i],
            'Hub': hub_vector[i]
        })
    
    # Creiamo un DataFrame a partire dalla lista generata
    df_risultati_hits = pd.DataFrame(risultati_combinati)

    # Aggiungiamo le colonne percentuali
    df_risultati_hits['Authority (%)'] = (df_risultati_hits['Authority'] * 100).round(2)
    df_risultati_hits['Hub (%)'] = (df_risultati_hits['Hub'] * 100).round(2)

    # Arrotondiamo i valori originali a 4 cifre
    df_risultati_hits['Authority'] = df_risultati_hits['Authority'].round(4)
    df_risultati_hits['Hub'] = df_risultati_hits['Hub'].round(4)

    # Riordiniamo le colonne per renderlo più facile da leggere
    df_risultati_hits = df_risultati_hits[['Nodo', 'Authority', 'Authority (%)', 'Hub', 'Hub (%)']]

    # Ordiniamo l'intero CSV in base al punteggio di Authority (dal più alto al più basso)
    df_risultati_hits = df_risultati_hits.sort_values(by='Authority', ascending=False)

    # Definiamo il nome del file di salvataggio
    nome_file_risultato = os.path.join(percorso_risultati, 'HITS_Classifica_Scenario1.csv')

    # Salviamo su disco
    df_risultati_hits.to_csv(nome_file_risultato, index=False) 
    print(f"\nFile HITS salvato correttamente in: {nome_file_risultato}") 

else:
    print(f"Errore: File {nome_file} non trovato. Controlla il percorso.")

--- ANALISI HITS: SCENARIO 1 ---
Caricamento del dataset: ../DataSet_CasoStudio1/Rete_100/dataset_scenario1.csv...
Generazione automatica della matrice di adiacenza L sparsa...

Avvio del calcolo HITS...
HITS Convergenza raggiunta all'iterazione 2.

Top 5 AUTHORITY (Chi riceve i link migliori):
1° Posto -> Nodo 0: Score 0.3333
2° Posto -> Nodo 1: Score 0.3333
3° Posto -> Nodo 2: Score 0.3333
4° Posto -> Nodo 3: Score 0.0000
5° Posto -> Nodo 4: Score 0.0000

Top 5 HUB (Chi smista i link migliori):
1° Posto -> Nodo 3: Score 0.0103
2° Posto -> Nodo 4: Score 0.0103
3° Posto -> Nodo 5: Score 0.0103
4° Posto -> Nodo 6: Score 0.0103
5° Posto -> Nodo 7: Score 0.0103

File HITS salvato correttamente in: ../Risultati_CasoStudio1_HITS/Rete_100/HITS_Classifica_Scenario1.csv


In [7]:
import os
import pandas as pd
import numpy as np

# =====================================================================
# TEST HITS SCENARIO 1 CON DATASET 1.000.000x1.000.000
# =====================================================================

numero_totale_nodi = 1000000 
print("--- ANALISI HITS: SCENARIO 1 (1 MILIONE DI NODI) ---")

# 1. PERCORSI CORRETTI (Input per Rete_1M)
nome_file = '../DataSet_CasoStudio1/Rete_1M/dataset_scenario1_1MILIONE.csv'
print(f"Caricamento del dataset: {nome_file}...")

# Verifichiamo che il file esista prima di procedere
if os.path.exists(nome_file):

    # 2. Leggiamo il CSV con Pandas
    df = pd.read_csv(nome_file)
    nodi_sorgente = df['Source'].values
    nodi_destinazione = df['Target'].values

    # 3. Generiamo la matrice di adiacenza sparsa L
    print("Generazione automatica della matrice di adiacenza L sparsa...")
    L_sparsa = prepara_matrice_adiacenza(nodi_sorgente, nodi_destinazione, numero_totale_nodi)

    # 4. Esecuzione dell'algoritmo HITS 
    print("\nAvvio del calcolo HITS...")
    authority_vector, hub_vector = hits_algorithm(L_sparsa, max_iter=1000)

    # =====================================================================
    # CREAZIONE DATAFRAME OTTIMIZZATA PER BIG DATA 
    # =====================================================================
    print("\nGenerazione veloce delle classifiche...")
    
    df_risultati_hits = pd.DataFrame({
        'Nodo': np.arange(numero_totale_nodi),
        'Authority': authority_vector,
        'Hub': hub_vector,
        'Authority (%)': (authority_vector * 100),
        'Hub (%)': (hub_vector * 100)
    })

    # Ordinamento doppio: in caso di pareggio di punteggio, ordina per Nodo crescente
    df_auth_sorted = df_risultati_hits.sort_values(by=['Authority', 'Nodo'], ascending=[False, True])
    df_hub_sorted = df_risultati_hits.sort_values(by=['Hub', 'Nodo'], ascending=[False, True])

    # STAMPA A SCHERMO CON MASSIMA PRECISIONE
    print("\nTop 5 AUTHORITY (Chi riceve i link migliori):")
    for pos, (idx, row) in enumerate(df_auth_sorted.head(5).iterrows()):
        print(f"{pos+1}° Posto -> Nodo {int(row['Nodo'])}: Score {row['Authority']:.4e} ({row['Authority (%)']:.6f}%)")

    print("\nTop 5 HUB (Chi smista i link migliori):")
    for pos, (idx, row) in enumerate(df_hub_sorted.head(5).iterrows()):
        print(f"{pos+1}° Posto -> Nodo {int(row['Nodo'])}: Score {row['Hub']:.4e} ({row['Hub (%)']:.6f}%)")

    # =====================================================================
    # SALVATAGGIO DEI RISULTATI NELLA SOTTOCARTELLA CORRETTA
    # =====================================================================
    percorso_risultati = '../Risultati_CasoStudio1_HITS/Rete_1M/'
    os.makedirs(percorso_risultati, exist_ok=True)

    # Arrotondamento di precisione: 8 cifre per gli score, 6 per le percentuali
    df_risultati_hits['Authority'] = df_risultati_hits['Authority'].round(8)
    df_risultati_hits['Hub'] = df_risultati_hits['Hub'].round(8)
    df_risultati_hits['Authority (%)'] = df_risultati_hits['Authority (%)'].round(6)
    df_risultati_hits['Hub (%)'] = df_risultati_hits['Hub (%)'].round(6)

    # Ordinamento finale pulito prima di salvare il CSV
    df_risultati_hits = df_risultati_hits.sort_values(by=['Authority', 'Nodo'], ascending=[False, True])

    nome_file_risultato = os.path.join(percorso_risultati, 'HITS_Classifica_Scenario1_1MILIONE.csv')
    df_risultati_hits.to_csv(nome_file_risultato, index=False) 
    
    print(f"\nFile HITS salvato correttamente in: {nome_file_risultato}") 

else:
    print(f"Errore: File {nome_file} non trovato. Controlla il percorso.")

--- ANALISI HITS: SCENARIO 1 (1 MILIONE DI NODI) ---
Caricamento del dataset: ../DataSet_CasoStudio1/Rete_1M/dataset_scenario1_1MILIONE.csv...
Generazione automatica della matrice di adiacenza L sparsa...

Avvio del calcolo HITS...
HITS Convergenza raggiunta all'iterazione 2.

Generazione veloce delle classifiche...

Top 5 AUTHORITY (Chi riceve i link migliori):
1° Posto -> Nodo 0: Score 3.3333e-01 (33.333333%)
2° Posto -> Nodo 1: Score 3.3333e-01 (33.333333%)
3° Posto -> Nodo 2: Score 3.3333e-01 (33.333333%)
4° Posto -> Nodo 3: Score 0.0000e+00 (0.000000%)
5° Posto -> Nodo 4: Score 0.0000e+00 (0.000000%)

Top 5 HUB (Chi smista i link migliori):
1° Posto -> Nodo 3: Score 1.0000e-06 (0.000100%)
2° Posto -> Nodo 4: Score 1.0000e-06 (0.000100%)
3° Posto -> Nodo 5: Score 1.0000e-06 (0.000100%)
4° Posto -> Nodo 6: Score 1.0000e-06 (0.000100%)
5° Posto -> Nodo 7: Score 1.0000e-06 (0.000100%)

File HITS salvato correttamente in: ../Risultati_CasoStudio1_HITS/Rete_1M/HITS_Classifica_Scenario1_

In [4]:
import os
import pandas as pd
import numpy as np

# =====================================================================
# TEST HITS SCENARIO 2 CON DATASET 100x100
# =====================================================================

numero_totale_nodi = 100 
print("--- ANALISI HITS: SCENARIO 2 ---")

# 1. Definiamo il file di input per lo scenario 2
nome_file_2 = '../DataSet_CasoStudio1/Rete_100/dataset_scenario2.csv'
print(f"Caricamento del dataset: {nome_file_2}...")

# Verifichiamo che il file esista prima di procedere
if os.path.exists(nome_file_2):

    # 2. Leggiamo il CSV con Pandas
    #inserisco il dataset in un DataFrame di Pandas per poterlo manipolare più facilmente
    df_2 = pd.read_csv(nome_file_2)

    # Estraiamo le colonne 'Source' e 'Target' come array numpy per costruire la matrice di adiacenza
     #  df['Source'].values ci dà un array numpy con i valori della colonna 'Source', che rappresentano i nodi di partenza dei link
     #  df['Target'].values ci dà un array numpy con i valori della colonna 'Target', che rappresentano i nodi di arrivo dei link
    nodi_sorgente_2 = df_2['Source'].values
    nodi_destinazione_2 = df_2['Target'].values

    # 3. Generiamo la matrice di adiacenza sparsa L (specifica per HITS)
    print("Generazione automatica della matrice di adiacenza L sparsa...")
    L_sparsa_2 = prepara_matrice_adiacenza(nodi_sorgente_2, nodi_destinazione_2, numero_totale_nodi)

    # 4. Esecuzione dell'algoritmo HITS
    print("\nAvvio del calcolo HITS...")
    authority_vector_2, hub_vector_2 = hits_algorithm(L_sparsa_2, max_iter=1000)

    # Output a schermo (stampiamo solo la Top 5 per brevità visiva)
    print("\nTop 5 AUTHORITY (Chi riceve i link migliori):")

    # Creiamo una lista di tuple (nodo, authority_score) e la ordiniamo in base al punteggio di authority
    classifica_auth_2 = [(i, authority_vector_2[i]) for i in range(numero_totale_nodi)]
    classifica_auth_2.sort(key=lambda x: x[1], reverse=True) # Ordiniamo in modo decrescente

    # Stampiamo i primi 5 nodi con i punteggi di authority più alti
    for pos, (nodo, auth) in enumerate(classifica_auth_2[:5]):
        print(f"{pos+1}° Posto -> Nodo {nodo}: Score {auth:.4f}")

    print("\nTop 5 HUB (Chi smista i link migliori):")

    # Creiamo una lista di tuple (nodo, hub_score) e la ordiniamo in base al punteggio di hub
    classifica_hub_2 = [(i, hub_vector_2[i]) for i in range(numero_totale_nodi)]
    classifica_hub_2.sort(key=lambda x: x[1], reverse=True) # Ordiniamo in modo decrescente

    # Stampiamo i primi 5 nodi con i punteggi di hub più alti
    for pos, (nodo, hub) in enumerate(classifica_hub_2[:5]):
        print(f"{pos+1}° Posto -> Nodo {nodo}: Score {hub:.4f}")

    # =====================================================================
    # SALVATAGGIO DEI RISULTATI NELLA CARTELLA 'Risultati_CasoStudio1_HITS'
    # =====================================================================
    percorso_risultati = '../Risultati_CasoStudio1_HITS/Rete_100/'

    # Se la cartella non esiste, creiamola
    if not os.path.exists(percorso_risultati):
        os.makedirs(percorso_risultati) #crea la cartella se non esiste
        print(f"\nCartella creata con successo in: {percorso_risultati}")

    # Creiamo un DataFrame "fuso" che contenga sia Hub che Authority per ogni nodo
    # Partiamo da una lista ordinata numericamente (dal Nodo 0 al Nodo 99)
    risultati_combinati_2 = []

    # Popoliamo la lista con i risultati di Authority e Hub per ogni nodo
    for i in range(numero_totale_nodi): 
        risultati_combinati_2.append({
            'Nodo': i,
            'Authority': authority_vector_2[i],
            'Hub': hub_vector_2[i]
        })
    
    # Creiamo un DataFrame a partire dalla lista generata
    df_risultati_hits_2 = pd.DataFrame(risultati_combinati_2) 

    # Aggiungiamo le colonne percentuali
    df_risultati_hits_2['Authority (%)'] = (df_risultati_hits_2['Authority'] * 100).round(2)
    df_risultati_hits_2['Hub (%)'] = (df_risultati_hits_2['Hub'] * 100).round(2)

    # Arrotondiamo i valori originali a 4 cifre
    df_risultati_hits_2['Authority'] = df_risultati_hits_2['Authority'].round(4)
    df_risultati_hits_2['Hub'] = df_risultati_hits_2['Hub'].round(4)

    # Riordiniamo le colonne per renderlo più facile da leggere
    df_risultati_hits_2 = df_risultati_hits_2[['Nodo', 'Authority', 'Authority (%)', 'Hub', 'Hub (%)']]

    # Ordiniamo l'intero CSV in base al punteggio di Authority (dal più alto al più basso)
    df_risultati_hits_2 = df_risultati_hits_2.sort_values(by='Authority', ascending=False)

    # Definiamo il nome del file di salvataggio (ATTENZIONE: Scenario 2)
    nome_file_risultato_2 = os.path.join(percorso_risultati, 'HITS_Classifica_Scenario2.csv')

    # Salviamo su disco
    df_risultati_hits_2.to_csv(nome_file_risultato_2, index=False) 
    print(f"\nFile HITS salvato correttamente in: {nome_file_risultato_2}") 

else:
    print(f"Errore: File {nome_file_2} non trovato. Controlla il percorso.")

--- ANALISI HITS: SCENARIO 2 ---
Caricamento del dataset: ../DataSet_CasoStudio1/Rete_100/dataset_scenario2.csv...
Generazione automatica della matrice di adiacenza L sparsa...

Avvio del calcolo HITS...
HITS Convergenza raggiunta all'iterazione 13.

Top 5 AUTHORITY (Chi riceve i link migliori):
1° Posto -> Nodo 1: Score 0.3078
2° Posto -> Nodo 2: Score 0.1719
3° Posto -> Nodo 0: Score 0.1718
4° Posto -> Nodo 78: Score 0.0123
5° Posto -> Nodo 74: Score 0.0122

Top 5 HUB (Chi smista i link migliori):
1° Posto -> Nodo 80: Score 0.0192
2° Posto -> Nodo 70: Score 0.0192
3° Posto -> Nodo 60: Score 0.0192
4° Posto -> Nodo 90: Score 0.0191
5° Posto -> Nodo 3: Score 0.0130

File HITS salvato correttamente in: ../Risultati_CasoStudio1_HITS/Rete_100/HITS_Classifica_Scenario2.csv


In [8]:
import os
import pandas as pd
import numpy as np

# =====================================================================
# TEST HITS SCENARIO 2 CON DATASET 1.000.000x1.000.000
# =====================================================================

numero_totale_nodi = 1000000 
print("--- ANALISI HITS: SCENARIO 2 (1 MILIONE DI NODI) ---")

nome_file_2 = '../DataSet_CasoStudio1/Rete_1M/dataset_scenario2_1MILIONE.csv'
print(f"Caricamento del dataset: {nome_file_2}...")

if os.path.exists(nome_file_2):

    df_2 = pd.read_csv(nome_file_2)
    nodi_sorgente_2 = df_2['Source'].values
    nodi_destinazione_2 = df_2['Target'].values

    print("Generazione automatica della matrice di adiacenza L sparsa...")
    L_sparsa_2 = prepara_matrice_adiacenza(nodi_sorgente_2, nodi_destinazione_2, numero_totale_nodi)

    print("\nAvvio del calcolo HITS...")
    authority_vector_2, hub_vector_2 = hits_algorithm(L_sparsa_2, max_iter=1000)

    # =====================================================================
    # CREAZIONE DATAFRAME OTTIMIZZATA PER BIG DATA
    # =====================================================================
    print("\nGenerazione veloce delle classifiche...")
    
    df_risultati_hits_2 = pd.DataFrame({
        'Nodo': np.arange(numero_totale_nodi),
        'Authority': authority_vector_2,
        'Hub': hub_vector_2,
        'Authority (%)': (authority_vector_2 * 100),
        'Hub (%)': (hub_vector_2 * 100)
    })

    # Ordinamento doppio: in caso di pareggio di punteggio, ordina per Nodo crescente
    df_auth_sorted_2 = df_risultati_hits_2.sort_values(by=['Authority', 'Nodo'], ascending=[False, True])
    df_hub_sorted_2 = df_risultati_hits_2.sort_values(by=['Hub', 'Nodo'], ascending=[False, True])

    print("\nTop 5 AUTHORITY (Chi riceve i link migliori):")
    for pos, (idx, row) in enumerate(df_auth_sorted_2.head(5).iterrows()):
        print(f"{pos+1}° Posto -> Nodo {int(row['Nodo'])}: Score {row['Authority']:.4e} ({row['Authority (%)']:.6f}%)")

    print("\nTop 5 HUB (Chi smista i link migliori):")
    for pos, (idx, row) in enumerate(df_hub_sorted_2.head(5).iterrows()):
        print(f"{pos+1}° Posto -> Nodo {int(row['Nodo'])}: Score {row['Hub']:.4e} ({row['Hub (%)']:.12f}%)")

    # =====================================================================
    # SALVATAGGIO DEI RISULTATI NELLA SOTTOCARTELLA CORRETTA
    # =====================================================================
    percorso_risultati = '../Risultati_CasoStudio1_HITS/Rete_1M/'
    os.makedirs(percorso_risultati, exist_ok=True)

    # Mettiamo 12 cifre anche nel salvataggio
    df_risultati_hits_2['Authority'] = df_risultati_hits_2['Authority'].round(12)
    df_risultati_hits_2['Hub'] = df_risultati_hits_2['Hub'].round(12)
    df_risultati_hits_2['Authority (%)'] = df_risultati_hits_2['Authority (%)'].round(10)
    df_risultati_hits_2['Hub (%)'] = df_risultati_hits_2['Hub (%)'].round(10)

    # Ordinamento finale pulito prima di salvare il CSV
    df_risultati_hits_2 = df_risultati_hits_2.sort_values(by=['Authority', 'Nodo'], ascending=[False, True])

    nome_file_risultato_2 = os.path.join(percorso_risultati, 'HITS_Classifica_Scenario2_1MILIONE.csv')
    df_risultati_hits_2.to_csv(nome_file_risultato_2, index=False) 
    
    print(f"\nFile HITS salvato correttamente in: {nome_file_risultato_2}") 

else:
    print(f"Errore: File {nome_file_2} non trovato. Controlla il percorso.")

--- ANALISI HITS: SCENARIO 2 (1 MILIONE DI NODI) ---
Caricamento del dataset: ../DataSet_CasoStudio1/Rete_1M/dataset_scenario2_1MILIONE.csv...
Generazione automatica della matrice di adiacenza L sparsa...

Avvio del calcolo HITS...
HITS Convergenza raggiunta all'iterazione 3.

Generazione veloce delle classifiche...

Top 5 AUTHORITY (Chi riceve i link migliori):
1° Posto -> Nodo 0: Score 2.4857e-01 (24.857057%)
2° Posto -> Nodo 1: Score 2.4857e-01 (24.857057%)
3° Posto -> Nodo 2: Score 2.4857e-01 (24.857057%)
4° Posto -> Nodo 811611: Score 2.7343e-06 (0.000273%)
5° Posto -> Nodo 509260: Score 2.7343e-06 (0.000273%)

Top 5 HUB (Chi smista i link migliori):
1° Posto -> Nodo 585000: Score 1.0000e-06 (0.000100003263%)
2° Posto -> Nodo 961500: Score 1.0000e-06 (0.000100003263%)
3° Posto -> Nodo 776000: Score 1.0000e-06 (0.000100003229%)
4° Posto -> Nodo 774000: Score 1.0000e-06 (0.000100003229%)
5° Posto -> Nodo 707000: Score 1.0000e-06 (0.000100003229%)

File HITS salvato correttamente in:

In [5]:
import os
import pandas as pd
import numpy as np

# =====================================================================
# TEST HITS SCENARIO 3 CON DATASET 100x100
# =====================================================================

numero_totale_nodi = 100 
print("--- ANALISI HITS: SCENARIO 3 ---")

# 1. Definiamo il file di input per lo scenario 3
nome_file_3 = '../DataSet_CasoStudio1/Rete_100/dataset_scenario3.csv'
print(f"Caricamento del dataset: {nome_file_3}...")

# Verifichiamo che il file esista prima di procedere
if os.path.exists(nome_file_3):

    # 2. Leggiamo il CSV con Pandas
    #inserisco il dataset in un DataFrame di Pandas per poterlo manipolare più facilmente
    df_3 = pd.read_csv(nome_file_3)

    # Estraiamo le colonne 'Source' e 'Target' come array numpy
     #  df['Source'].values ci dà un array numpy con i valori della colonna 'Source', che rappresentano i nodi di partenza dei link
     # df['Target'].values ci dà un array numpy con i valori della colonna 'Target', che rappresentano i nodi di arrivo dei link    
    nodi_sorgente_3 = df_3['Source'].values
    nodi_destinazione_3 = df_3['Target'].values

    # 3. Generiamo la matrice di adiacenza sparsa L (specifica per HITS)
    print("Generazione automatica della matrice di adiacenza L sparsa...")
    L_sparsa_3 = prepara_matrice_adiacenza(nodi_sorgente_3, nodi_destinazione_3, numero_totale_nodi)

    # 4. Esecuzione dell'algoritmo HITS
    print("\nAvvio del calcolo HITS...")
    authority_vector_3, hub_vector_3 = hits_algorithm(L_sparsa_3, max_iter=1000)

    # Output a schermo (stampiamo solo la Top 5 per brevità visiva)
    print("\nTop 5 AUTHORITY (Chi riceve i link migliori):")

    # Creiamo una lista di tuple (nodo, authority_score) e la ordiniamo in base al punteggio di authority
    classifica_auth_3 = [(i, authority_vector_3[i]) for i in range(numero_totale_nodi)]
    classifica_auth_3.sort(key=lambda x: x[1], reverse=True)

    # Stampiamo i primi 5 nodi con i punteggi di authority più alti
    for pos, (nodo, auth) in enumerate(classifica_auth_3[:5]):
        print(f"{pos+1}° Posto -> Nodo {nodo}: Score {auth:.4f}")

     # Output a schermo (stampiamo solo la Top 5 per brevità visiva)
    print("\nTop 5 HUB (Chi smista i link migliori):")

    # Creiamo una lista di tuple (nodo, hub_score) e la ordiniamo in base al punteggio di hub
    classifica_hub_3 = [(i, hub_vector_3[i]) for i in range(numero_totale_nodi)]
    classifica_hub_3.sort(key=lambda x: x[1], reverse=True)

    # Stampiamo i primi 5 nodi con i punteggi di hub più alti
    for pos, (nodo, hub) in enumerate(classifica_hub_3[:5]):
        print(f"{pos+1}° Posto -> Nodo {nodo}: Score {hub:.4f}")

    # =====================================================================
    # SALVATAGGIO DEI RISULTATI NELLA CARTELLA 'Risultati_CasoStudio1_HITS'
    # =====================================================================
    percorso_risultati = '../Risultati_CasoStudio1_HITS/Rete_100/'

    # Se la cartella non esiste, creiamola
    if not os.path.exists(percorso_risultati):
        os.makedirs(percorso_risultati)
        print(f"\nCartella creata con successo in: {percorso_risultati}")

    # Creiamo un DataFrame "fuso" che contenga sia Hub che Authority per ogni nodo
    risultati_combinati_3 = []

    # Popoliamo la lista con i risultati di Authority e Hub per ogni nodo
    for i in range(numero_totale_nodi):
        risultati_combinati_3.append({
            'Nodo': i,
            'Authority': authority_vector_3[i],
            'Hub': hub_vector_3[i]
        })
    
    # Creiamo un DataFrame a partire dalla lista generata
    df_risultati_hits_3 = pd.DataFrame(risultati_combinati_3)

    # Aggiungiamo le colonne percentuali
    df_risultati_hits_3['Authority (%)'] = (df_risultati_hits_3['Authority'] * 100).round(2)
    df_risultati_hits_3['Hub (%)'] = (df_risultati_hits_3['Hub'] * 100).round(2)

    # Arrotondiamo i valori originali a 4 cifre
    df_risultati_hits_3['Authority'] = df_risultati_hits_3['Authority'].round(4)
    df_risultati_hits_3['Hub'] = df_risultati_hits_3['Hub'].round(4)

    # Riordiniamo le colonne per renderlo più facile da leggere
    df_risultati_hits_3 = df_risultati_hits_3[['Nodo', 'Authority', 'Authority (%)', 'Hub', 'Hub (%)']]

    # Ordiniamo l'intero CSV in base al punteggio di Authority (dal più alto al più basso)
    df_risultati_hits_3 = df_risultati_hits_3.sort_values(by='Authority', ascending=False)

    # Definiamo il nome del file di salvataggio (ATTENZIONE: Scenario 3)
    nome_file_risultato_3 = os.path.join(percorso_risultati, 'HITS_Classifica_Scenario3.csv')

    # Salviamo su disco
    df_risultati_hits_3.to_csv(nome_file_risultato_3, index=False) 
    print(f"\nFile HITS salvato correttamente in: {nome_file_risultato_3}") 

else:
    print(f"Errore: File {nome_file_3} non trovato. Controlla il percorso.")

--- ANALISI HITS: SCENARIO 3 ---
Caricamento del dataset: ../DataSet_CasoStudio1/Rete_100/dataset_scenario3.csv...
Generazione automatica della matrice di adiacenza L sparsa...

Avvio del calcolo HITS...
HITS Convergenza raggiunta all'iterazione 87.

Top 5 AUTHORITY (Chi riceve i link migliori):
1° Posto -> Nodo 2: Score 0.1891
2° Posto -> Nodo 0: Score 0.1407
3° Posto -> Nodo 1: Score 0.1383
4° Posto -> Nodo 80: Score 0.0214
5° Posto -> Nodo 20: Score 0.0212

Top 5 HUB (Chi smista i link migliori):
1° Posto -> Nodo 80: Score 0.0224
2° Posto -> Nodo 50: Score 0.0214
3° Posto -> Nodo 40: Score 0.0214
4° Posto -> Nodo 70: Score 0.0203
5° Posto -> Nodo 20: Score 0.0192

File HITS salvato correttamente in: ../Risultati_CasoStudio1_HITS/Rete_100/HITS_Classifica_Scenario3.csv


In [9]:
import os
import pandas as pd
import numpy as np

# =====================================================================
# TEST HITS SCENARIO 3 CON DATASET 1.000.000x1.000.000
# =====================================================================

numero_totale_nodi = 1000000 
print("--- ANALISI HITS: SCENARIO 3 (1 MILIONE DI NODI) ---")

nome_file_3 = '../DataSet_CasoStudio1/Rete_1M/dataset_scenario3_1MILIONE.csv'
print(f"Caricamento del dataset: {nome_file_3}...")

if os.path.exists(nome_file_3):

    df_3 = pd.read_csv(nome_file_3)
    nodi_sorgente_3 = df_3['Source'].values
    nodi_destinazione_3 = df_3['Target'].values

    print("Generazione automatica della matrice di adiacenza L sparsa...")
    L_sparsa_3 = prepara_matrice_adiacenza(nodi_sorgente_3, nodi_destinazione_3, numero_totale_nodi)

    print("\nAvvio del calcolo HITS...")
    authority_vector_3, hub_vector_3 = hits_algorithm(L_sparsa_3, max_iter=1000)

    # =====================================================================
    # CREAZIONE DATAFRAME OTTIMIZZATA PER BIG DATA
    # =====================================================================
    print("\nGenerazione veloce delle classifiche...")
    
    df_risultati_hits_3 = pd.DataFrame({
        'Nodo': np.arange(numero_totale_nodi),
        'Authority': authority_vector_3,
        'Hub': hub_vector_3,
        'Authority (%)': (authority_vector_3 * 100),
        'Hub (%)': (hub_vector_3 * 100)
    })

    # Ordinamento doppio: in caso di pareggio di punteggio, ordina per Nodo crescente
    df_auth_sorted_3 = df_risultati_hits_3.sort_values(by=['Authority', 'Nodo'], ascending=[False, True])
    df_hub_sorted_3 = df_risultati_hits_3.sort_values(by=['Hub', 'Nodo'], ascending=[False, True])

    print("\nTop 5 AUTHORITY (Chi riceve i link migliori):")
    for pos, (idx, row) in enumerate(df_auth_sorted_3.head(5).iterrows()):
        print(f"{pos+1}° Posto -> Nodo {int(row['Nodo'])}: Score {row['Authority']:.4e} ({row['Authority (%)']:.6f}%)")

    print("\nTop 5 HUB (Chi smista i link migliori):")
    for pos, (idx, row) in enumerate(df_hub_sorted_3.head(5).iterrows()):
        print(f"{pos+1}° Posto -> Nodo {int(row['Nodo'])}: Score {row['Hub']:.4e} ({row['Hub (%)']:.6f}%)")

    # =====================================================================
    # SALVATAGGIO DEI RISULTATI NELLA SOTTOCARTELLA CORRETTA
    # =====================================================================
    percorso_risultati = '../Risultati_CasoStudio1_HITS/Rete_1M/'
    os.makedirs(percorso_risultati, exist_ok=True)

    df_risultati_hits_3['Authority'] = df_risultati_hits_3['Authority'].round(8)
    df_risultati_hits_3['Hub'] = df_risultati_hits_3['Hub'].round(8)
    df_risultati_hits_3['Authority (%)'] = df_risultati_hits_3['Authority (%)'].round(6)
    df_risultati_hits_3['Hub (%)'] = df_risultati_hits_3['Hub (%)'].round(6)

    # Ordinamento finale pulito prima di salvare il CSV
    df_risultati_hits_3 = df_risultati_hits_3.sort_values(by=['Authority', 'Nodo'], ascending=[False, True])

    nome_file_risultato_3 = os.path.join(percorso_risultati, 'HITS_Classifica_Scenario3_1MILIONE.csv')
    df_risultati_hits_3.to_csv(nome_file_risultato_3, index=False) 
    
    print(f"\nFile HITS salvato correttamente in: {nome_file_risultato_3}") 

else:
    print(f"Errore: File {nome_file_3} non trovato. Controlla il percorso.")

--- ANALISI HITS: SCENARIO 3 (1 MILIONE DI NODI) ---
Caricamento del dataset: ../DataSet_CasoStudio1/Rete_1M/dataset_scenario3_1MILIONE.csv...
Generazione automatica della matrice di adiacenza L sparsa...

Avvio del calcolo HITS...
HITS Convergenza raggiunta all'iterazione 10.

Generazione veloce delle classifiche...

Top 5 AUTHORITY (Chi riceve i link migliori):
1° Posto -> Nodo 0: Score 2.5461e-01 (25.461456%)
2° Posto -> Nodo 1: Score 2.1384e-01 (21.383995%)
3° Posto -> Nodo 2: Score 1.5729e-01 (15.729018%)
4° Posto -> Nodo 594000: Score 1.6870e-05 (0.001687%)
5° Posto -> Nodo 203000: Score 1.6513e-05 (0.001651%)

Top 5 HUB (Chi smista i link migliori):
1° Posto -> Nodo 593999: Score 1.3673e-06 (0.000137%)
2° Posto -> Nodo 639999: Score 1.3673e-06 (0.000137%)
3° Posto -> Nodo 479999: Score 1.3673e-06 (0.000137%)
4° Posto -> Nodo 504999: Score 1.3673e-06 (0.000137%)
5° Posto -> Nodo 781002: Score 1.3673e-06 (0.000137%)

File HITS salvato correttamente in: ../Risultati_CasoStudio1_HIT